# 4 · The crystal scale

With `particle_model="crystal"` every cathode volume holds a crystal in which lithium diffuses (`docs/model.md` section 12), instead of a particle of uniform concentration. The crystal is a sphere by default (or a cylinder or slab), with radius `R_p` and solid diffusivity `D_c`. The crystals and the electrode are solved together in one Newton iteration per time step. Parameters are illustrative defaults, not fitted to a cell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from lfp_model.crystal import CrystalModel
from lfp_model.params import Params
from lfp_model.simulate import run

to_mAhg = lambda p, eq: eq * p.F / (p.M * 3.6)
crystal = lambda **kw: Params(mode="corrected", particle_model="crystal", **kw)

## Uniform particles and crystals

At the default D_c = 8 × 10⁻¹⁴ cm²/s, the diffusion time R_p²/D_c of a 200 nm crystal is about 5000 s, comparable to a 1C discharge, so solid diffusion costs capacity.

In [ ]:
pu, pc = Params(mode="corrected", C_rate=1.0), crystal(C_rate=1.0)
au, ac = run(pu).array, run(pc).array
plt.plot(to_mAhg(pu, au[:, 2]), au[:, 1], label="uniform particles")
plt.plot(to_mAhg(pc, ac[:, 2]), ac[:, 1], label="crystals, D_c = 8e-14 cm²/s")
plt.xlabel("capacity [mAh/g]"); plt.ylabel("cell voltage [V]"); plt.title("1C discharge"); plt.legend(); plt.show()

## Inside the crystals

Lithium profiles θ(r) after 1800 s at 1C, in crystals near the separator, in the middle of the electrode, and near the current collector. Each crystal fills from its surface (θ ≈ 0.6 at the surface, 0.36 at the centre). The three profiles nearly coincide: in this thin electrode, transport in the electrolyte and the solid phase is fast, and diffusion inside the crystals is what limits.

In [ ]:
p = crystal(C_rate=1.0)
m = CrystalModel(p)
st = run(p, max_steps=1800).final_state
theta = m.cs(st) / m.cs_max
r_nm = m.cm.r * 1e7
for l, where in ((0, "near the separator"), (m.nl // 2, "middle"), (m.nl - 1, "near the collector")):
    plt.plot(r_nm, theta[l], "o-", ms=3, label=where)
plt.xlabel("r [nm]"); plt.ylabel("θ"); plt.title("crystal profiles after 1800 s at 1C"); plt.legend(); plt.show()

## Capacity against rate and solid diffusivity

In [ ]:
rates = [1.0, 2.0, 5.0]
for D_c in (1e-12, 8e-14, 1e-14):
    caps = [to_mAhg(crystal(), run(crystal(C_rate=cr, D_c=D_c)).array[-1, 2]) for cr in rates]
    plt.semilogx(rates, caps, "o-", label=f"D_c = {D_c:g} cm²/s")
plt.xlabel("C-rate"); plt.ylabel("capacity to 2.5 V [mAh/g]"); plt.legend(); plt.show()

## The uniform limit

As D_c grows the profiles flatten and the crystal model approaches the uniform-particle model: the voltage difference falls like 1/D_c.

In [ ]:
a_u = run(Params(mode="corrected", C_rate=1.0), max_steps=900).array
Ds = [1e-12, 1e-11, 1e-10, 1e-9]
dV = []
for D_c in Ds:
    a_c = run(crystal(C_rate=1.0, D_c=D_c), max_steps=900).array
    n = min(len(a_u), len(a_c))
    dV.append(np.abs(a_c[:n, 1] - a_u[:n, 1]).max())
plt.loglog(Ds, dV, "o-", label="crystal − uniform")
plt.loglog(Ds, dV[0] * Ds[0] / np.array(Ds), "k--", lw=0.8, label="∝ 1/D_c")
plt.xlabel("D_c [cm²/s]"); plt.ylabel("max |ΔV| over 900 s at 1C [V]"); plt.legend(); plt.show()

## Crystal shape

At the same R_p a sphere has three times, and a cylinder twice, the surface per volume of a slab, and a shorter mean diffusion path.

In [ ]:
for shape in ("sphere", "cylinder", "slab"):
    p = crystal(C_rate=2.0, crystal_shape=shape)
    a = run(p).array
    plt.plot(to_mAhg(p, a[:, 2]), a[:, 1], label=shape)
plt.xlabel("capacity [mAh/g]"); plt.ylabel("cell voltage [V]"); plt.title("2C discharge"); plt.legend(); plt.show()